# Automated IFC Door Scheduling and Information Quality Checks

## Summary

Learn how to extract architectural door data from an IFC model, define project information requirements, check classifications and dimensions automatically, and export the results as an Excel quality report.

**Target audience:** OpenBIM Analyst – Level 3 (basic Python knowledge recommended).

**Learning outcomes:** You will be able to (1) inspect IFC door data, (2) distinguish IFC entities from classification references, (3) define information requirements, and (4) run and interpret automated checks.

**Before starting:** Place `26-01-D-ARCH.ifc` in the same folder as this notebook. You will need Python, IfcOpenShell, Pandas and openpyxl. Run the cells **from top to bottom**.

**Scope:** This is an *information-quality* exercise. The classification rule and door width limits are examples for this tutorial, not official building regulations or proof of legal compliance.

## Step 1 – Load the IFC model

IfcOpenShell opens the IFC file, and `by_type("IfcDoor")` retrieves its door elements. We use this model throughout the tutorial.

**Expected:** 130 doors in `26-01-D-ARCH.ifc`.

In [1]:
from pathlib import Path
import ifcopenshell

model_path = Path("26-01-D-ARCH.ifc")
if not model_path.is_file():
    raise FileNotFoundError(f"Put {model_path.name} in the notebook folder.")

model = ifcopenshell.open(str(model_path))
doors = model.by_type("IfcDoor")

print("IFC schema:", model.schema)
print("Doors found:", len(doors))

IFC schema: IFC4X3
Doors found: 130


## Step 2 – Extract dimensions and classification references

Each `IfcDoor` has identifiers and may have dimensions and classification references. First, we convert IFC length units to **millimetres**, rather than assuming the model already uses millimetres.

`IfcDoor` describes an **IFC entity type**. *Uniformat* is a separate **classification system**. A classification may be assigned directly to a door or inherited from its door type; `get_references()` checks these references.

**Expected:** A preview of five doors and the number of doors with classification references.

In [2]:
import pandas as pd
import ifcopenshell.util.unit
import ifcopenshell.util.classification

# Convert the model's length unit to millimetres
unit_to_mm = ifcopenshell.util.unit.calculate_unit_scale(model) * 1000

# Read each IFC door once and retain its classification system/code pairs
records = []
for door in doors:
    pairs = []
    for ref in ifcopenshell.util.classification.get_references(door):
        system = ifcopenshell.util.classification.get_classification(ref)
        system_name = (system.Name or "").strip() if system else ""
        code = (ref.Identification or "").strip()
        pairs.append((system_name, code))

    pairs = sorted(set(pairs))
    records.append({
        "Door ID": door.GlobalId,
        "Door Name": door.Name or "",
        "Width (mm)": round(door.OverallWidth * unit_to_mm, 1) if door.OverallWidth is not None else None,
        "Height (mm)": round(door.OverallHeight * unit_to_mm, 1) if door.OverallHeight is not None else None,
        "Classification": "; ".join(f"{system}: {code}" for system, code in pairs),
        "_pairs": pairs,  # Internal data used for the validation rule
    })

door_df = pd.DataFrame(records)
print("Doors with classification:", door_df["_pairs"].map(bool).sum())
display(door_df.drop(columns="_pairs").head())

Doors with classification: 12


,Door ID,Door Name,Width (mm),Height (mm),Classification
0,1XB8_BDLj9KRf58am8PuHf,M_Door-Passage-Single-Flush:750 x 2000mm:1670997,850.0,2100.0,
1,1XB8_BDLj9KRf58am8PuHI,M_Door-Passage-Single-Flush:750 x 2000mm:1671022,850.0,2100.0,
2,1XB8_BDLj9KRf58am8PuJS,M_Door-Passage-Single-Flush:750 x 2000mm:1671136,850.0,2100.0,
3,1XB8_BDLj9KRf58am8PtFP,M_Door-Passage-Single-Flush:750 x 2000mm:1673445,850.0,2100.0,
4,1XB8_BDLj9KRf58am8Pt9z,M_Door-Passage-Single-Flush:750 x 2000mm:1673537,850.0,2100.0,


## Step 3 – Define project information requirements

We will check that a door has a **name**, **positive recorded dimensions**, and a classification reference matching a specified **system and code**.

We also flag unusually narrow or wide recorded values for **manual review**. These thresholds (500–3000 mm) are deliberately chosen for this exercise; they are **not** universal minimum or maximum legal door widths.

**Try later:** Change `MIN_WIDTH_MM` during the practical exercise.

In [9]:
REQUIRED_SYSTEM = "Uniformat"
ALLOWED_CODES = {"C1020110"}

# Teaching example: unusual values trigger review, not automatic non-compliance
MIN_WIDTH_MM = 500
MAX_WIDTH_MM = 3000

print("Required classification:", REQUIRED_SYSTEM, ALLOWED_CODES)
print("Width review range:", MIN_WIDTH_MM, "to", MAX_WIDTH_MM, "mm")

Required classification: Uniformat {'C1020110'}
Width review range: 500 to 3000 mm


## Step 4 – Check each door against the requirements

This step turns the requirements into rules. **FAIL** means a required item is missing or does not match our chosen rule. **REVIEW** identifies an unusual width worth checking manually; a review flag is not proof that the door is incorrect.

The system and code must match **within the same classification reference**. A code being on the allowed list is not by itself proof that it is the most appropriate classification for the real-world door.

**Expected for the original requirements:** 130 doors, 12 PASS, 118 FAIL, and one unusual-width review.

In [10]:
results = []

for _, door in door_df.iterrows():
    issues = []
    name = door["Door Name"].strip()
    width = door["Width (mm)"]
    height = door["Height (mm)"]
    pairs = door["_pairs"]

    # Required information
    if not name:
        issues.append("Missing name")
    if pd.isna(width) or width <= 0:
        issues.append("Missing or invalid width")
    if pd.isna(height) or height <= 0:
        issues.append("Missing or invalid height")

    # Match classification system and code in the same reference
    if not pairs:
        issues.append("Missing classification")
    elif not any(
        system.casefold() == REQUIRED_SYSTEM.casefold() and code in ALLOWED_CODES
        for system, code in pairs
    ):
        issues.append("Wrong classification system or code")

    # An unusual positive width is a review flag, not an automatic failure
    if pd.isna(width) or width <= 0:
        dimension_check = "FAIL: Missing or invalid width"
    elif width < MIN_WIDTH_MM or width > MAX_WIDTH_MM:
        dimension_check = "REVIEW: Unusual width"
    else:
        dimension_check = "OK"

    results.append({
        "Door ID": door["Door ID"],
        "Door Name": door["Door Name"],
        "Width (mm)": width,
        "Height (mm)": height,
        "Classification": door["Classification"],
        "Status": "FAIL" if issues else "PASS",
        "Issues": "; ".join(issues) if issues else "None",
        "Dimension Check": dimension_check,
    })

report_df = pd.DataFrame(results)
print("Total doors:", len(report_df))
print("PASS:", (report_df["Status"] == "PASS").sum())
print("FAIL:", (report_df["Status"] == "FAIL").sum())
print("Dimension reviews:", (report_df["Dimension Check"] != "OK").sum())
display(report_df.head())

Total doors: 130
PASS: 12
FAIL: 118
Dimension reviews: 1


,Door ID,Door Name,Width (mm),Height (mm),Classification,Status,Issues,Dimension Check
0,1XB8_BDLj9KRf58am8PuHf,M_Door-Passage-Single-Flush:750 x 2000mm:1670997,850.0,2100.0,,FAIL,Missing classification,OK
1,1XB8_BDLj9KRf58am8PuHI,M_Door-Passage-Single-Flush:750 x 2000mm:1671022,850.0,2100.0,,FAIL,Missing classification,OK
2,1XB8_BDLj9KRf58am8PuJS,M_Door-Passage-Single-Flush:750 x 2000mm:1671136,850.0,2100.0,,FAIL,Missing classification,OK
3,1XB8_BDLj9KRf58am8PtFP,M_Door-Passage-Single-Flush:750 x 2000mm:1673445,850.0,2100.0,,FAIL,Missing classification,OK
4,1XB8_BDLj9KRf58am8Pt9z,M_Door-Passage-Single-Flush:750 x 2000mm:1673537,850.0,2100.0,,FAIL,Missing classification,OK


## Step 5 – Export the Excel quality report

The report has three sheets: **Door Schedule** (all doors), **Quality Issues** (failed required checks), and **Dimension Review** (unusual or invalid widths). The same door can appear on more than one sheet.

The Excel file is saved in the notebook folder.

In [11]:
quality_issues_df = report_df[report_df["Status"] == "FAIL"].copy()
dimension_review_df = report_df[report_df["Dimension Check"] != "OK"].copy()

report_file = "door_quality_report_final.xlsx"
with pd.ExcelWriter(report_file, engine="openpyxl") as writer:
    report_df.to_excel(writer, sheet_name="Door Schedule", index=False)
    quality_issues_df.to_excel(writer, sheet_name="Quality Issues", index=False)
    dimension_review_df.to_excel(writer, sheet_name="Dimension Review", index=False)

print("Created:", report_file)
print("Door Schedule:", len(report_df))
print("Quality Issues:", len(quality_issues_df))
print("Dimension Review:", len(dimension_review_df))

Created: door_quality_report_final.xlsx
Door Schedule: 130
Quality Issues: 118
Dimension Review: 1


## Practical exercise – Change a quality rule

1. Run the notebook once and note the numbers of **PASS**, **FAIL**, and **Dimension Review** doors.
2. In **Step 3**, change `MIN_WIDTH_MM` from `500` to `900`. Re-run **Steps 3–5**.
3. How many additional doors are flagged for dimension review? Why does the number of `FAIL` doors stay the same or change?
4. Restore `MIN_WIDTH_MM = 500` and re-run **Steps 3–5** to return to the baseline report.